# Handling Missing Values in Environmental Data
Eight small examples corresponding to common missing-value situations.

## 1. Only a few missing values → Row removal

In [ ]:
import pandas as pd
df = pd.read_csv("01_few_missing_row_removal.csv")
print("Before:", df.shape)
df_clean = df.dropna()
print("After:", df_clean.shape)
df_clean


## 2. Numeric missing values → Mean or median imputation

In [ ]:
df = pd.read_csv("02_numeric_mean_median_imputation.csv")
df["mean_imputed"] = df["soil_moisture_pct"].fillna(df["soil_moisture_pct"].mean())
df["median_imputed"] = df["soil_moisture_pct"].fillna(df["soil_moisture_pct"].median())
df


## 3. Categorical missing values → Mode or 'Unknown'

In [ ]:
df = pd.read_csv("03_categorical_mode_unknown.csv")
mode_value = df["land_use"].mode()[0]
df["mode_imputed"] = df["land_use"].fillna(mode_value)
df["unknown_category"] = df["land_use"].fillna("Unknown")
df


## 4. Time-series missing values → Interpolation and Kalman smoothing

In [ ]:
!pip install -q pykalman

import numpy as np
from pykalman import KalmanFilter

df = pd.read_csv("04_timeseries_interpolation_kalman.csv", parse_dates=["datetime"])

# Linear interpolation
df["linear_interpolation"] = df["temperature_C"].interpolate()

# Kalman smoothing
values = df["temperature_C"].to_numpy(dtype=float)
masked_values = np.ma.masked_invalid(values)

kf = KalmanFilter(
    initial_state_mean=np.nanmean(values),
    initial_state_covariance=1.0,
    observation_covariance=1.0,
    transition_covariance=0.1
)

state_means, _ = kf.smooth(masked_values)
df["kalman_smoothing"] = state_means.ravel()
df


## 5. Group-related missingness → Group-wise imputation

In [ ]:
df = pd.read_csv("05_groupwise_imputation.csv")
group_median = df.groupby("land_use")["soil_moisture_pct"].transform("median")
df["groupwise_imputed"] = df["soil_moisture_pct"].fillna(group_median)
df


## 6. High missingness → Remove variable or recollect data

In [ ]:
df = pd.read_csv("06_high_missingness_remove_variable.csv")
missing_ratio = df.isna().mean() * 100
print(missing_ratio)

threshold = 50
cols_to_drop = missing_ratio[missing_ratio > threshold].index
df_reduced = df.drop(columns=cols_to_drop)

print("Dropped columns:", list(cols_to_drop))
df_reduced


## 7. Measurement/recording errors → Check the original source

In [ ]:
df = pd.read_csv("07_measurement_recording_error.csv")
error_rows = df[df["temperature_C"].isna() | df["rainfall_mm"].isna()]
error_rows


## 8. Non-random missingness → Investigate the cause first

In [ ]:
df = pd.read_csv("08_nonrandom_missingness_investigate.csv")
df["temp_missing"] = df["temperature_C"].isna()

print("Missingness by road access:")
display(pd.crosstab(df["road_access"], df["temp_missing"], normalize="index"))

print("Missing rate by winter sampling availability:")
display(df.groupby("winter_sample_available")["temp_missing"].mean())
